In [10]:
# imports

import pandas as pd
import numpy as np
import json
import joblib
from pathlib import Path

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

In [12]:
# load dataset

DATA_PATH = Path("../data/processed/ham10000_clean.csv")
SAVE_DIR = Path("../data/processed")
SAVE_DIR.mkdir(exist_ok=True)

df = pd.read_csv(DATA_PATH)

print("Loaded dataset:", df.shape)
df.head()

Loaded dataset: (10015, 10)


,lesion_id,image_id,dx,dx_type,age,sex,localization,image_path,width,height
0,HAM_0000118,ISIC_0027419,bkl,histo,80.0,male,scalp,/home/saber/Wox/ACV/Case_Study/data/raw/ham100...,600,450
1,HAM_0000118,ISIC_0025030,bkl,histo,80.0,male,scalp,/home/saber/Wox/ACV/Case_Study/data/raw/ham100...,600,450
2,HAM_0002730,ISIC_0026769,bkl,histo,80.0,male,scalp,/home/saber/Wox/ACV/Case_Study/data/raw/ham100...,600,450
3,HAM_0002730,ISIC_0025661,bkl,histo,80.0,male,scalp,/home/saber/Wox/ACV/Case_Study/data/raw/ham100...,600,450
4,HAM_0001466,ISIC_0031633,bkl,histo,75.0,male,ear,/home/saber/Wox/ACV/Case_Study/data/raw/ham100...,600,450


In [13]:
# drop dx_type column if it exists, since it's not needed for training and can cause issues with the model

if "dx_type" in df.columns:
    df = df.drop(columns=["dx_type"])

print("Columns after cleanup:")
print(df.columns)

Columns after cleanup:
Index(['lesion_id', 'image_id', 'dx', 'age', 'sex', 'localization',
       'image_path', 'width', 'height'],
      dtype='str')


In [14]:
# label encoding 7-class

le = LabelEncoder()
df["label"] = le.fit_transform(df["dx"])

label_map = {
    str(cls): int(idx)
    for cls, idx in zip(le.classes_, le.transform(le.classes_))
}

print("Label Map:")
print(label_map)

Label Map:
{'akiec': 0, 'bcc': 1, 'bkl': 2, 'df': 3, 'mel': 4, 'nv': 5, 'vasc': 6}


In [16]:
# metadata processing

#age

df["age"] = df["age"].fillna(df["age"].median())

# sex

df["sex"] = df["sex"].map({
    "male": 0,
    "female": 1
})

df["sex"] = df["sex"].fillna(-1)

# localization

df = pd.get_dummies(df, columns=["localization"])

In [17]:
df.dtypes

lesion_id                           str
image_id                            str
dx                                  str
age                             float64
sex                             float64
image_path                          str
width                             int64
height                            int64
label                             int64
localization_abdomen               bool
localization_acral                 bool
localization_back                  bool
localization_chest                 bool
localization_ear                   bool
localization_face                  bool
localization_foot                  bool
localization_genital               bool
localization_hand                  bool
localization_lower extremity       bool
localization_neck                  bool
localization_scalp                 bool
localization_trunk                 bool
localization_unknown               bool
localization_upper extremity       bool
dtype: object

In [18]:
# convert bool to float

for col in df.columns:
    if df[col].dtype == "bool":
        df[col] = df[col].astype(float)

In [19]:
df.dtypes

lesion_id                           str
image_id                            str
dx                                  str
age                             float64
sex                             float64
image_path                          str
width                             int64
height                            int64
label                             int64
localization_abdomen            float64
localization_acral              float64
localization_back               float64
localization_chest              float64
localization_ear                float64
localization_face               float64
localization_foot               float64
localization_genital            float64
localization_hand               float64
localization_lower extremity    float64
localization_neck               float64
localization_scalp              float64
localization_trunk              float64
localization_unknown            float64
localization_upper extremity    float64
dtype: object

In [20]:
# train-val split

unique_lesions = df["lesion_id"].unique()

train_lesions, val_lesions = train_test_split(
    unique_lesions,
    test_size=0.2,
    random_state=42
)

train_df = df[df["lesion_id"].isin(train_lesions)].copy()
val_df = df[df["lesion_id"].isin(val_lesions)].copy()

print("Train images:", len(train_df))
print("Val images:", len(val_df))
print("Train lesions:", train_df["lesion_id"].nunique())
print("Val lesions:", val_df["lesion_id"].nunique())


Train images: 7974
Val images: 2041
Train lesions: 5976
Val lesions: 1494


In [21]:
# sanity check

overlap = set(train_df["lesion_id"]).intersection(set(val_df["lesion_id"]))
print("Lesion overlap:", len(overlap))

Lesion overlap: 0


In [22]:
# normalize age

scaler = StandardScaler()

train_df["age"] = scaler.fit_transform(train_df[["age"]])
val_df["age"] = scaler.transform(val_df[["age"]])

In [24]:
# compute class weights

class_weights_array = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(train_df["label"]),
    y=train_df["label"]
)

class_weights_list = [float(w) for w in class_weights_array]

class_weights_list

[4.398234969663541,
 2.9820493642483172,
 1.3323308270676693,
 14.063492063492063,
 1.2828185328185329,
 0.2108352502577933,
 10.746630727762803]

In [25]:
# save data

# Save train and validation CSV
train_df.to_csv(SAVE_DIR / "train.csv", index=False)
val_df.to_csv(SAVE_DIR / "val.csv", index=False)

# Save label map
with open(SAVE_DIR / "label_map.json", "w") as f:
    json.dump(label_map, f, indent=4)

# Save class weights
with open(SAVE_DIR / "class_weights.json", "w") as f:
    json.dump(class_weights_list, f, indent=4)

# Save age scaler
joblib.dump(scaler, SAVE_DIR / "age_scaler.pkl")

print("All artifacts saved successfully.")

All artifacts saved successfully.
